In [1]:
# first thing that our rag will need is data

with open("/content/miniragv1text.txt", encoding="utf-8") as file:
    content = file.read()

In [2]:
print(content)

Neural Radiance Fields (NeRF) represent a cutting-edge technique in 3D computer graphics that uses neural networks to generate novel views of complex 3D scenes. Unlike traditional graphics pipelines that rely on explicit geometry and texture maps, NeRF models learn an implicit volumetric representation of a scene by optimizing a continuous function that maps spatial coordinates and viewing directions to color and volume density. This function is parameterized by a multilayer perceptron (MLP), allowing the scene to be rendered with photorealistic detail and consistent lighting effects.

One of the core innovations in NeRF is the use of volumetric rendering, where a ray is cast from the camera into the scene and sampled at multiple depths. At each sample point, the network predicts both the color and the density, and the final pixel color is computed as a weighted integration along the ray. This rendering technique enables the capture of complex visual phenomena like soft shadows, semi-t

In [3]:
#next we would need to create chunks
#we are choosing fixed size chunking with overlap of 5 words
#the langchain character text splitter is not working, so we just implemented a simpler version of it.
# Use a simple chunking function instead of the broken langchain dependency chain.
def chunk_text(text, chunk_size=50, chunk_overlap=5):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start += chunk_size - chunk_overlap

    return chunks

chunks = chunk_text(content, chunk_size=50, chunk_overlap=5)
print(f"Total chunks: {len(chunks)}")


Total chunks: 12


In [4]:
print(chunks[0])
print(chunks[1])

Neural Radiance Fields (NeRF) represent a cutting-edge technique in 3D computer graphics that uses neural networks to generate novel views of complex 3D scenes. Unlike traditional graphics pipelines that rely on explicit geometry and texture maps, NeRF models learn an implicit volumetric representation of a scene by optimizing a continuous
scene by optimizing a continuous function that maps spatial coordinates and viewing directions to color and volume density. This function is parameterized by a multilayer perceptron (MLP), allowing the scene to be rendered with photorealistic detail and consistent lighting effects. One of the core innovations in NeRF is the use


In [5]:
#next step would be to create embedings for the chunks.
#usually in a rag, we store the genrated embedings in a vector database, but for this example we will just use a list to store the embedings.
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')
sentence_embeddings = embedder.encode(chunks)
print(f"Total embeddings: {len(sentence_embeddings)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total embeddings: 12


In [7]:
print(sentence_embeddings.shape)


(12, 384)


In [9]:
#now are task is to fetch top 3 chunks based on our query
#we would first write a query, create its embeding, right now sentence embedding kind of is our dummy vectordb

query ="What is Neural Radiance field?"
query_embedding = embedder.encode([query])
print(query_embedding.shape)

(1, 384)


In [14]:
import numpy as np
from numpy.linalg import norm
def cosinesimlarity(a,b):
  # Ensure 'a' is a 1D array for scalar output
  if a.ndim > 1:
    a = a.flatten()
  dot_prod =np.dot(a,b)
  norm_a = norm(a)
  norm_b = norm(b)
  return dot_prod/(norm_a*norm_b)

In [16]:
#we would want to calculate simalrity score for query embedding against each chunk embedding
score=[]
for embedding in sentence_embeddings:
  score.append(cosinesimlarity(query_embedding,embedding))
#then we would sort this array
print("Original scores:", score)
#we would want to sort the score array
sorted_scores = sorted(score, reverse=True)
print("Sorted scores (descending):")
print(sorted_scores)

Original scores: [np.float32(0.5570026), np.float32(0.2732711), np.float32(0.27627543), np.float32(0.23755625), np.float32(0.10160485), np.float32(0.19217268), np.float32(0.21152596), np.float32(0.17968911), np.float32(0.13589807), np.float32(0.26607838), np.float32(0.12840001), np.float32(0.13775405)]
Sorted scores (descending):
[np.float32(0.5570026), np.float32(0.27627543), np.float32(0.2732711), np.float32(0.26607838), np.float32(0.23755625), np.float32(0.21152596), np.float32(0.19217268), np.float32(0.17968911), np.float32(0.13775405), np.float32(0.13589807), np.float32(0.12840001), np.float32(0.10160485)]


In [18]:
#now we would want to fetch top 3 chunks
def get_top_chunks(scores, chunks, k=3):
    top_indices = np.argsort(scores)[-k:][::-1]

    return [(chunks[i], scores[i]) for i in top_indices]

In [19]:
results = get_top_chunks(score, chunks)

for chunk, similarity in results:
    print(similarity, chunk)

0.5570026 Neural Radiance Fields (NeRF) represent a cutting-edge technique in 3D computer graphics that uses neural networks to generate novel views of complex 3D scenes. Unlike traditional graphics pipelines that rely on explicit geometry and texture maps, NeRF models learn an implicit volumetric representation of a scene by optimizing a continuous
0.27627543 in NeRF is the use of volumetric rendering, where a ray is cast from the camera into the scene and sampled at multiple depths. At each sample point, the network predicts both the color and the density, and the final pixel color is computed as a weighted integration along the
0.2732711 scene by optimizing a continuous function that maps spatial coordinates and viewing directions to color and volume density. This function is parameterized by a multilayer perceptron (MLP), allowing the scene to be rendered with photorealistic detail and consistent lighting effects. One of the core innovations in NeRF is the use


In [20]:
# the above was simple technique to get the top 3 simlar chunk to query
#in rag we do not give entire document as context in the prompt, we give only the most relevant part, so in a way we have already implemented the retrieval part, next is augmented generation.

context = context = "\n\n".join([chunk for chunk, score in results])
print(context)


Neural Radiance Fields (NeRF) represent a cutting-edge technique in 3D computer graphics that uses neural networks to generate novel views of complex 3D scenes. Unlike traditional graphics pipelines that rely on explicit geometry and texture maps, NeRF models learn an implicit volumetric representation of a scene by optimizing a continuous

in NeRF is the use of volumetric rendering, where a ray is cast from the camera into the scene and sampled at multiple depths. At each sample point, the network predicts both the color and the density, and the final pixel color is computed as a weighted integration along the

scene by optimizing a continuous function that maps spatial coordinates and viewing directions to color and volume density. This function is parameterized by a multilayer perceptron (MLP), allowing the scene to be rendered with photorealistic detail and consistent lighting effects. One of the core innovations in NeRF is the use


In [22]:
#building prompt
prompt = f"""
Answer the user questions based on the provided context
Context: {context}
Question: {query}
Answer:
"""
print(prompt)


Answer the user questions based on the provided context
Context: Neural Radiance Fields (NeRF) represent a cutting-edge technique in 3D computer graphics that uses neural networks to generate novel views of complex 3D scenes. Unlike traditional graphics pipelines that rely on explicit geometry and texture maps, NeRF models learn an implicit volumetric representation of a scene by optimizing a continuous

in NeRF is the use of volumetric rendering, where a ray is cast from the camera into the scene and sampled at multiple depths. At each sample point, the network predicts both the color and the density, and the final pixel color is computed as a weighted integration along the

scene by optimizing a continuous function that maps spatial coordinates and viewing directions to color and volume density. This function is parameterized by a multilayer perceptron (MLP), allowing the scene to be rendered with photorealistic detail and consistent lighting effects. One of the core innovations in 

In [26]:
# Install the Groq SDK
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.6 MB/s eta 0:00:00


In [28]:
# Used to securely store your API key
from google.colab import userdata

# Configure Groq API key
GROQ_API_KEY = userdata.get('GROQ_API_KEY')


In [29]:
from groq import Groq

client = Groq(
    api_key=GROQ_API_KEY,
)

In [31]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="llama-3.1-8b-instant", # Updated to a currently supported Groq model
)

print(response.choices[0].message.content)

A Neural Radiance Field (NeRF) is a cutting-edge technique in 3D computer graphics that uses neural networks to generate novel views of complex 3D scenes. It represents a scene using a continuous function that maps spatial coordinates and viewing directions to color and volume density, allowing for photorealistic rendering with consistent lighting effects.
